# §8.2 운동 시간대 회귀 검증 노트북

Phase 1 §8.1·§2.1 위에서 추가된 `recommend_workout_window`가 *LLM 환각 시간대를 데이터 기반*으로 대체하는지 검증.

## 가설

1. **사용자별로 다른 권장 시간대가 나와야 한다** — 모든 회원이 `(20:00, 20:30)`로 수렴하는 LLM 환각 패턴 회피.
2. **회귀 추천 슬롯의 *그날 efficiency 평균*이 cohort default (`COHORT_DEFAULT_SLOT`) 슬롯의 평균보다 높다** — 회귀가 의미 있는 신호 학습.
3. **데이터 부족(< 14일)일 땐 cohort default fallback이 작동한다** — cold-start 안전.

## 실행 전제

Phase 1과 동일: `docker compose up -d` + 회원 CSV 적재 후 jupyter 실행. vLLM 미사용.

## Setup

In [ ]:
import os, sys
sys.path.insert(0, '../ai_service')

import numpy as np
import pandas as pd
from sleep_coach_full_kr_v6 import (
    build_master, recommend_workout_window,
    add_roll7, KEYS, TARGET, LEAK_VARS, COHORT_DEFAULT_SLOT, MODEL_VERSION
)
from db_utils import read_table

UID = os.getenv('AUDIT_UID', '23RK3S')
print(f'[setup] uid={UID} | MODEL_VERSION={MODEL_VERSION}')

In [ ]:
# Phase 1 누수 제거 입력 위에서 작동 확인 — train_master 빌드
master = add_roll7(build_master(UID))
master[TARGET] = master[TARGET].shift(-1)
train_master = master.dropna(subset=[TARGET])
print(f'[train_master] shape={train_master.shape} | date range={train_master["date"].min()} ~ {train_master["date"].max()}')
print(f'[train_master] efficiency mean={train_master[TARGET].mean():.2f} | std={train_master[TARGET].std():.2f}')

## 시나리오 1 — 시간대별 활동 분포 시각화 (전체 일자 기준)

In [ ]:
# 회원의 모든 일자에 대한 시간대별 평균 steps
from sleep_coach_full_kr_v6 import _combine_dt
df_min = read_table(f'{UID}_activity_1min', where=f"user_id = '{UID}'")
df_min['timestamp'] = _combine_dt(df_min)
df_min['date'] = df_min['timestamp'].dt.normalize()
df_min['hour'] = df_min['timestamp'].dt.hour
all_hours_steps = df_min.groupby('hour')['steps'].sum().sort_index()
print('[all-day] hourly steps:')
print(all_hours_steps)

## 시나리오 2 — 효율 상위 30% 일자만 필터한 시간대별 활동

*수면 효율이 좋았던 날*에 어떤 시간대에 운동했는지 분포가 *전체 분포*와 다른가 — 회귀 신호의 본질.

In [ ]:
eff_threshold = float(train_master[TARGET].quantile(0.7))
good_days = train_master.loc[train_master[TARGET] >= eff_threshold, 'date']
good_dates = pd.to_datetime(good_days).dt.normalize().unique()
print(f'[good-eff] threshold={eff_threshold:.2f} | days={len(good_dates)}')

df_good = df_min[df_min['date'].isin(good_dates)]
good_hours_steps = df_good.groupby('hour')['steps'].sum().sort_index()
comparison = pd.DataFrame({'all_days': all_hours_steps, 'good_eff_days': good_hours_steps}).fillna(0)
comparison['delta_pct'] = ((comparison['good_eff_days'] / comparison['good_eff_days'].sum()) -
                            (comparison['all_days'] / comparison['all_days'].sum())) * 100
comparison.sort_values('delta_pct', ascending=False).head(10)

## 시나리오 3 — recommend_workout_window 출력 ⭐

본 함수가 main()에서 호출되어 LLM 프롬프트의 시간 슬롯을 채움. 출력이 *cohort default*가 아닌 회원 맞춤이어야 1번 가설 충족.

In [ ]:
slots = recommend_workout_window(UID, train_master, k=3)
print(f'[§8.2] uid={UID} → top-3 slots = {slots}')
is_cohort_fallback = (len(slots) == 1 and slots[0] == COHORT_DEFAULT_SLOT)
print(f'[§8.2] cohort_fallback = {is_cohort_fallback}')

## 시나리오 4 — 회귀 슬롯 vs cohort default — efficiency 평균 비교

*가설 2*: 회귀가 추천한 슬롯에서 운동한 날의 efficiency 평균이 cohort default 슬롯에서 운동한 날의 평균보다 높아야 함.

In [ ]:
def avg_eff_for_slot(slot, df_min, train_master):
    """slot=(start_str,end_str) — 그 시간대에 운동(steps>0)했던 날의 train_master efficiency 평균."""
    s_h = int(slot[0].split(':')[0])
    e_h = int(slot[1].split(':')[0])
    df_in_slot = df_min[(df_min['hour'] >= s_h) & (df_min['hour'] < e_h) & (df_min['steps'] > 0)]
    active_dates = df_in_slot['date'].unique()
    eff_active = train_master[train_master['date'].isin(active_dates)][TARGET]
    return float(eff_active.mean()) if len(eff_active) > 0 else None

rec_eff = avg_eff_for_slot(slots[0], df_min, train_master) if not is_cohort_fallback else None
cohort_eff = avg_eff_for_slot(COHORT_DEFAULT_SLOT, df_min, train_master)
print(f'[비교] 회귀 추천 슬롯 {slots[0]} → eff 평균 = {rec_eff}')
print(f'[비교] cohort default {COHORT_DEFAULT_SLOT} → eff 평균 = {cohort_eff}')
if rec_eff and cohort_eff:
    diff = rec_eff - cohort_eff
    print(f'[비교] 회귀 - cohort = {diff:+.2f} ({"회귀 우위" if diff > 0 else "cohort 우위"})')

## 시나리오 5 — 데이터 부족 시 fallback 검증 (가설 3)

train_master를 인위적으로 13일로 잘라 fallback이 작동하는지 확인.

In [ ]:
short_master = train_master.head(13).copy()
fallback_slots = recommend_workout_window(UID, short_master, k=3)
expected = [COHORT_DEFAULT_SLOT]
ok = (fallback_slots == expected)
print(f'[fallback] short_master={len(short_master)}d → slots={fallback_slots}')
print(f'[fallback] expected={expected} | match={ok}')
assert ok, 'fallback should return cohort default for <14 days'

## 비교 표

In [ ]:
summary = pd.DataFrame([
    {'check': '가설 1 — 사용자 맞춤 슬롯 (cohort 아님)', 'result': not is_cohort_fallback},
    {'check': '가설 2 — 회귀 슬롯 eff > cohort eff',
     'result': (rec_eff or 0) > (cohort_eff or 0) if rec_eff and cohort_eff else None},
    {'check': '가설 3 — fallback 작동 (<14d)', 'result': ok},
])
summary

## Conclusion

### 합격선 해석

1. **가설 1 — 사용자 맞춤성**: 회귀가 회원의 활동 패턴을 반영해 cohort 외 슬롯을 산출하면 ✅. 만약 *cohort fallback*이 나오면 데이터가 < 14일이거나 효율 상위 일자에 활동이 없는 케이스 — 안전 동작.

2. **가설 2 — 회귀 우위**: 회귀 슬롯의 평균 efficiency가 cohort default보다 높으면 ✅. 차이가 작거나 음수면 *현재 features와 데이터로는 회귀 신호 미약* — 후속 §3.2(주관·객관 격차), §4(within-subject 실험)으로 진정한 인과 신호 보강 필요.

3. **가설 3 — fallback**: assert로 통과 강제. 운영 단계에서 신규 회원 cold-start의 안전 보장.

### PR/면접 답변용 한 줄

> "운동 시간대 추천을 LLM 환각이 아닌 *회원의 효율 상위 일자 활동 분포*에서 회귀해 산출. 데이터 부족 시 cohort default로 fallback. 회귀 슬롯의 efficiency 평균이 cohort 대비 X% 우위 (시나리오 4 출력 참조). 진정한 인과 검증은 §4 within-subject 실험으로 이어집니다."

### 후속 과제

- **§4 within-subject 미니 실험**: 노트북 출력 슬롯을 실제로 *따랐을 때 vs 평소대로 운동했을 때*의 효율 차이를 paired t-test로 검증
- **§3.2 주관·객관 격차 모델**: efficiency가 같아도 *체감*이 다르면 슬롯 추천에 가중치 차등
- **§8.5 prompt_hash 활용**: 같은 회원·같은 슬롯 추천에서도 LLM 메시지가 다른 빈도 추적해 *온도/프롬프트 안정성* 분석